In [ ]:
from langchain_community.graph_vectorstores import CassandraGraphVectorStore
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.graph_vectorstores.extractors import (
    LinkExtractorTransformer,
    KeybertLinkExtractor
)
from langchain_community.document_loaders import PyPDFLoader, WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.tools import tool
from typing import List, Dict, Optional
import os

# Set up environment variables
os.environ["OPENAI_API_KEY"] = "your-openai-api-key"
os.environ["LANGCHAIN_API_KEY"] = "your-langchain-api-key"  # if using LangSmith
os.environ["LANGCHAIN_TRACING_V2"] = "true"  # Enable tracing if needed

class GraphRAGSystem:
    def __init__(self):
        """Initialize the Graph RAG system with all necessary components."""
        # Initialize embeddings
        self.embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        # Initialize Graph Vector Store
        self.graph_vector_store = CassandraGraphVectorStore(self.embeddings)

        # Initialize LLM
        self.llm = ChatOpenAI(model="gpt-4-turbo", temperature=0)

        # Initialize text splitter
        self.text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200,
            length_function=len,
            is_separator_regex=False,
        )

        # Initialize link extractor pipeline
        self.link_extractor = LinkExtractorTransformer([
            KeybertLinkExtractor(top_n=3)  # Extract top 3 keywords per document
        ])

        # Set up RAG chain
        self.setup_rag_chain()

        # Set up agent
        self.setup_agent()

    def setup_rag_chain(self):
        """Set up the RAG chain with graph-enhanced retrieval."""
        # Define the hybrid retriever
        retriever = self.graph_vector_store.as_retriever(
            search_type="hybrid",  # Combine vector and graph search
            search_kwargs={
                "k": 6,           # Number of initial documents to retrieve
                "depth": 2,        # How far to traverse the graph
                "score_threshold": 0.7  # Minimum similarity score
            }
        )

        # Define the prompt template
        template = """You are an expert assistant with access to a knowledge graph.
        Answer the question based on the following context, which includes information
        from semantically similar documents and their graph connections:

        Context:
        {context}

        Question: {question}

        Provide a comprehensive answer that synthesizes information across connected documents.
        If uncertain, say so and suggest possible related queries."""

        prompt = ChatPromptTemplate.from_template(template)

        # Create the RAG chain with additional metadata
        self.rag_chain = (
            {"context": retriever, "question": RunnablePassthrough()}
            | prompt
            | self.llm
            | StrOutputParser()
        )

    def setup_agent(self):
        """Set up the agent with tools for document management and search."""
        @tool
        def add_documents(file_paths: List[str]) -> Dict:
            """Add documents from file paths or URLs to the vector store.
            Supports PDFs and web pages. Returns processing statistics."""
            documents = []
            for path in file_paths:
                try:
                    if path.startswith(("http://", "https://")):
                        loader = WebBaseLoader(path)
                        docs = loader.load()
                        docs[0].metadata["source"] = path  # Ensure URL is preserved
                    else:
                        loader = PyPDFLoader(path)
                        docs = loader.load()

                    # Split and extract links
                    split_docs = self.text_splitter.split_documents(docs)
                    processed_docs = self.link_extractor.transform_documents(split_docs)
                    documents.extend(processed_docs)
                except Exception as e:
                    print(f"Error processing {path}: {str(e)}")
                    continue

            # Add to vector store
            if documents:
                doc_ids = self.graph_vector_store.add_documents(documents)
                return {
                    "status": "success",
                    "documents_added": len(doc_ids),
                    "sample_doc_id": doc_ids[0],
                    "warnings": f"{len(file_paths) - len(documents)} failures"
                               if len(file_paths) != len(documents) else None
                }
            return {"status": "failed", "reason": "No valid documents processed"}

        @tool
        def graph_search(query: str, k: int = 5, depth: int = 2) -> Dict:
            """Perform a graph-aware vector search. Returns documents and their connections.

            Args:
                query: The search query
                k: Number of initial documents to retrieve
                depth: How many hops to traverse in the graph
            """
            results = self.graph_vector_store.traversal_search(
                query=query,
                k=k,
                depth=depth,
                return_connections=True
            )

            # Format results
            formatted = {
                "documents": [],
                "connection_graph": {}
            }

            for doc in results["documents"]:
                formatted["documents"].append({
                    "content": doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content,
                    "metadata": doc.metadata,
                    "id": doc.metadata.get("id", "unknown")
                })

            for source, targets in results["connections"].items():
                formatted["connection_graph"][source] = {
                    "connected_to": len(targets),
                    "sample_connections": list(targets)[:3]  # Show first 3
                }

            return formatted

        @tool
        def answer_question(question: str, detailed: bool = False) -> Dict:
            """Answer a question using the RAG system with graph-enhanced retrieval.

            Args:
                question: The question to answer
                detailed: Whether to include retrieval metadata
            """
            answer = self.rag_chain.invoke(question)

            result = {"answer": answer}

            if detailed:
                # Get additional retrieval details
                if hasattr(self.rag_chain, 'last_retrieval'):
                    retrieval = self.rag_chain.last_retrieval
                    result["retrieval_metadata"] = {
                        "documents_retrieved": len(retrieval["documents"]),
                        "graph_traversal_depth": retrieval["depth"],
                        "average_similarity_score": f"{retrieval['avg_score']:.2f}"
                    }

            return result

        # Define tools
        tools = [add_documents, graph_search, answer_question]

        # Define agent prompt
        agent_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are a Graph RAG Assistant with these capabilities:
            1. Add documents to the knowledge graph (PDFs or URLs)
            2. Perform graph-aware searches
            3. Answer questions with context from connected documents

            Always verify document additions were successful.
            For searches, recommend appropriate depth based on query complexity.
            For answers, synthesize information across connected documents."""),
            ("human", "{input}"),
            ("placeholder", "{agent_scratchpad}")
        ])

        # Create agent
        agent = create_tool_calling_agent(self.llm, tools, agent_prompt)
        self.agent_executor = AgentExecutor(
            agent=agent,
            tools=tools,
            verbose=True,
            handle_parsing_errors=True,
            max_iterations=5
        )

    def query(self, input_text: str) -> str:
        """Main interface to query the agentic RAG system."""
        try:
            result = self.agent_executor.invoke({"input": input_text})
            return result["output"]
        except Exception as e:
            return f"Error processing query: {str(e)}"

def display_results(result: Dict, indent: int = 2) -> None:
    """Helper function to display nested results."""
    if isinstance(result, dict):
        for key, value in result.items():
            if isinstance(value, dict):
                print(" " * indent + f"{key}:")
                display_results(value, indent + 2)
            elif isinstance(value, list):
                print(" " * indent + f"{key}:")
                for i, item in enumerate(value, 1):
                    print(" " * (indent + 2) + f"{i}. {str(item)[:100]}...")
            else:
                print(" " * indent + f"{key}: {value}")
    else:
        print(" " * indent + str(result))

if __name__ == "__main__":
    print("""
    ██████╗  ██████╗  █████╗ ██████╗ ██╗  ██╗     ██████╗ █████╗  ██████╗
    ██╔══██╗██╔════╝ ██╔══██╗██╔══██╗██║  ██║    ██╔════╝██╔══██╗██╔════╝
    ██████╔╝██║  ███╗███████║██████╔╝███████║    ██║     ███████║██║  ███╗
    ██╔══██╗██║   ██║██╔══██║██╔═══╝ ██╔══██║    ██║     ██╔══██║██║   ██║
    ██║  ██║╚██████╔╝██║  ██║██║     ██║  ██║    ╚██████╗██║  ██║╚██████╔╝
    ╚═╝  ╚═╝ ╚═════╝ ╚═╝  ╚═╝╚═╝     ╚═╝  ╚═╝     ╚═════╝╚═╝  ╚═╝ ╚═════╝
    """)
    print("Initializing Graph RAG System with Knowledge Graph capabilities...\n")

    rag_system = GraphRAGSystem()
    print("System initialized successfully!")
    print("Embedding model: OpenAI text-embedding-3-small")
    print("LLM: GPT-4-turbo")
    print("Vector store: Cassandra with graph extensions\n")

    # Sample documents for demonstration
    sample_documents = [
        "https://en.wikipedia.org/wiki/Knowledge_graph",
        "https://arxiv.org/pdf/2005.11401.pdf",  # RAG paper
        "https://www.ncbi.nlm.nih.gov/pmc/articles/PMC7727689/"  # Medical KG example
    ]

    while True:
        print("\n" + "="*50)
        print("MAIN MENU".center(50))
        print("="*50)
        print("1. Add documents to knowledge graph")
        print("2. Ask a question with RAG")
        print("3. Explore knowledge graph")
        print("4. System information")
        print("5. Exit")

        choice = input("\nEnter your choice (1-5): ").strip()

        if choice == "1":
            print("\n" + "-"*20 + " DOCUMENT INGESTION " + "-"*20)
            print("\nYou can add:")
            print("- Local PDF files (provide path)")
            print("- Web pages (provide URL)")
            print("- Multiple documents (comma separated)")

            print(f"\nSample documents available:\n{sample_documents}")
            doc_input = input("\nEnter document paths/URLs (or press Enter for samples): ").strip()

            doc_paths = [path.strip() for path in doc_input.split(",")] if doc_input else sample_documents

            print(f"\nProcessing {len(doc_paths)} documents...")
            result = rag_system.agent_executor.invoke({
                "input": "Add these documents to the knowledge graph",
                "file_paths": doc_paths
            })

            print("\nProcessing Results:")
            display_results(result["output"])

            # Show knowledge graph stats if available
            try:
                stats = rag_system.graph_vector_store.get_graph_stats()
                print("\nKnowledge Graph Updated:")
                print(f"- Total nodes: {stats['nodes']}")
                print(f"- Total edges: {stats['edges']}")
                print(f"- New connections created: {stats.get('new_edges', 'N/A')}")
            except Exception as e:
                print(f"\nCould not retrieve graph stats: {str(e)}")

        elif choice == "2":
            print("\n" + "-"*20 + " QUESTION ANSWERING " + "-"*20)
            question = input("\nEnter your question (or press Enter for sample): ").strip()
            if not question:
                question = "How do knowledge graphs enhance RAG systems?"
                print(f"Using sample question: {question}")

            detailed = input("Include retrieval details? (y/n): ").lower() == 'y'

            print("\nProcessing question with graph-enhanced RAG...")
            result = rag_system.agent_executor.invoke({
                "input": f"Answer this question with {'detailed ' if detailed else ''}context: {question}",
                "detailed": detailed
            })

            print("\n" + "="*50)
            print("ANSWER".center(50))
            print("="*50)
            display_results(result["output"])

        elif choice == "3":
            print("\n" + "-"*20 + " GRAPH EXPLORATION " + "-"*20)
            query = input("\nEnter search query (or press Enter for sample): ").strip()
            if not query:
                query = "knowledge graph applications"
                print(f"Using sample query: {query}")

            k = input("Number of initial documents to retrieve (default 5): ").strip()
            depth = input("Traversal depth (default 2): ").strip()

            try:
                k = int(k) if k else 5
                depth = int(depth) if depth else 2
            except ValueError:
                print("Invalid input, using defaults")
                k, depth = 5, 2

            print(f"\nSearching with k={k}, depth={depth}...")
            result = rag_system.agent_executor.invoke({
                "input": f"Explore the knowledge graph for: {query}",
                "k": k,
                "depth": depth
            })

            print("\n" + "="*50)
            print("GRAPH EXPLORATION RESULTS".center(50))
            print("="*50)
            display_results(result["output"])

        elif choice == "4":
            print("\n" + "="*50)
            print("SYSTEM INFORMATION".center(50))
            print("="*50)
            print("Graph RAG System Configuration:")
            print("- Vector Store: Cassandra with graph extensions")
            print("- Embedding Model: OpenAI text-embedding-3-small")
            print("- LLM: GPT-4-turbo")
            print("- Text Splitting: RecursiveCharacterTextSplitter (1000/200)")
            print("- Link Extraction: Keybert keyword extraction")

            try:
                stats = rag_system.graph_vector_store.get_graph_stats()
                print("\nCurrent Knowledge Graph Stats:")
                print(f"- Documents: {stats['nodes']}")
                print(f"- Connections: {stats['edges']}")
                print(f"- Avg connections per node: {stats.get('avg_connections', 'N/A')}")
            except Exception as e:
                print(f"\nCould not retrieve graph stats: {str(e)}")

        elif choice == "5":
            print("\nExiting Graph RAG System. Goodbye!")
            break

        else:
            print("Invalid choice. Please enter a number between 1-5.")

        input("\nPress Enter to continue...")